# Projeto 1 - Cena de Pesca 3D

In [276]:
%pip install pyopengl --break-system-packages
%pip install glfw --break-system-packages
%pip install PyGLM  --break-system-packages

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.
Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.
Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [277]:
import glfw
from OpenGL.GL import *
import numpy as np
import glm
import math

In [278]:
glfw.init()
glfw.window_hint(glfw.VISIBLE, glfw.FALSE)
window = glfw.create_window(700, 700, "Cena de Pesca", None, None)
glfw.make_context_current(window)

In [279]:
vertex_code = """
        attribute vec3 position;
        uniform mat4 mat_transformation;
        void main(){
            gl_Position = mat_transformation * vec4(position,1.0);
        }
        """

fragment_code = """
        uniform vec4 color;
        void main(){
            gl_FragColor = color;
        }
        """

In [280]:
def create_shader_program(vertex_source, fragment_source):
    program  = glCreateProgram()
    vertex   = compile_shader(vertex_source, GL_VERTEX_SHADER)
    fragment = compile_shader(fragment_source, GL_FRAGMENT_SHADER)

    glAttachShader(program, vertex)
    glAttachShader(program, fragment)

    glLinkProgram(program)
    if not glGetProgramiv(program, GL_LINK_STATUS):
        raise RuntimeError(f"Erro de linkagem: {glGetProgramInfoLog(program).decode()}")

    return program

def compile_shader(source, shader_type):
    shader = glCreateShader(shader_type)
    glShaderSource(shader, source)
    glCompileShader(shader)
    
    if not glGetShaderiv(shader, GL_COMPILE_STATUS):
        error = glGetShaderInfoLog(shader).decode()
        tipo = "Vertex" if shader_type == GL_VERTEX_SHADER else "Fragment"
        raise RuntimeError(f"Erro de compilacao do {tipo} Shader: {error}")
        
    return shader

program = create_shader_program(vertex_code, fragment_code)
glUseProgram(program)

In [281]:
# Colocando a camera no ponto (3, 4, 3) e olhando para a origem, com FOV de 30 graus
view = glm.lookAt(glm.vec3(3, 4, 3), glm.vec3(0, 0, 0), glm.vec3(0, 1, 0))
projection = glm.perspective(glm.radians(30.0), 1.0, 0.1, 100.0)
mat_vp = projection * view

In [282]:
def gerar_circulo(raio, n_segmentos, centro_y=0.0):
    """Gera triangulos de um disco no plano xz, centrado na origem."""
    vertices = []
    
    for indice_segmento in range(n_segmentos):
        angulo_inicial = 2.0 * math.pi * indice_segmento / n_segmentos
        angulo_final = 2.0 * math.pi * (indice_segmento + 1) / n_segmentos
        
        x_inicial = raio * math.cos(angulo_inicial)
        z_inicial = raio * math.sin(angulo_inicial)
        
        x_final = raio * math.cos(angulo_final)
        z_final = raio * math.sin(angulo_final)

        vertices.append((0.0, centro_y, 0.0))
        vertices.append((x_inicial, centro_y, z_inicial))
        vertices.append((x_final, centro_y, z_final))
        
    return vertices


def gerar_cilindro(raio, altura, n_segmentos, y_base=0.0):
    """Gera triangulos de um cilindro no eixo Y."""
    vertices = []
    y_topo = y_base + altura
    
    for indice_segmento in range(n_segmentos):
        angulo_inicial = 2.0 * math.pi * indice_segmento / n_segmentos
        angulo_final = 2.0 * math.pi * (indice_segmento + 1) / n_segmentos
        
        x_inicial = raio * math.cos(angulo_inicial)
        z_inicial = raio * math.sin(angulo_inicial)
        
        x_final = raio * math.cos(angulo_final)
        z_final = raio * math.sin(angulo_final)
        
        # Primeiro triângulo da face lateral
        vertices.append((x_inicial, y_base, z_inicial))
        vertices.append((x_final, y_base, z_final))
        vertices.append((x_inicial, y_topo, z_inicial))
        
        # Segundo triângulo da face lateral
        vertices.append((x_final, y_base, z_final))
        vertices.append((x_final, y_topo, z_final))
        vertices.append((x_inicial, y_topo, z_inicial))
        
    vertices += gerar_circulo(raio, n_segmentos, y_topo)
    vertices += gerar_circulo(raio, n_segmentos, y_base)
    
    return vertices


def gerar_esfera(raio, n_latitudes, n_longitudes, centro_y=0.0):
    """Gera triangulos de uma esfera centrada em (0, centro_y, 0)."""
    vertices = []
    
    for indice_latitude in range(n_latitudes):
        latitude_inicial = math.pi * indice_latitude / n_latitudes - math.pi / 2
        latitude_final = math.pi * (indice_latitude + 1) / n_latitudes - math.pi / 2
        
        for indice_longitude in range(n_longitudes):
            longitude_inicial = 2.0 * math.pi * indice_longitude / n_longitudes
            longitude_final = 2.0 * math.pi * (indice_longitude + 1) / n_longitudes

            # Ponto 1: Inferior Esquerdo
            x_inf_esq = raio * math.cos(latitude_inicial) * math.cos(longitude_inicial)
            y_inf_esq = raio * math.sin(latitude_inicial) + centro_y
            z_inf_esq = raio * math.cos(latitude_inicial) * math.sin(longitude_inicial)

            # Ponto 2: Superior Esquerdo
            x_sup_esq = raio * math.cos(latitude_final) * math.cos(longitude_inicial)
            y_sup_esq = raio * math.sin(latitude_final) + centro_y
            z_sup_esq = raio * math.cos(latitude_final) * math.sin(longitude_inicial)

            # Ponto 3: Superior Direito
            x_sup_dir = raio * math.cos(latitude_final) * math.cos(longitude_final)
            y_sup_dir = raio * math.sin(latitude_final) + centro_y
            z_sup_dir = raio * math.cos(latitude_final) * math.sin(longitude_final)

            # Ponto 4: Inferior Direito
            x_inf_dir = raio * math.cos(latitude_inicial) * math.cos(longitude_final)
            y_inf_dir = raio * math.sin(latitude_inicial) + centro_y
            z_inf_dir = raio * math.cos(latitude_inicial) * math.sin(longitude_final)

            # Triângulo 1
            vertices.append((x_inf_esq, y_inf_esq, z_inf_esq))
            vertices.append((x_sup_esq, y_sup_esq, z_sup_esq))
            vertices.append((x_sup_dir, y_sup_dir, z_sup_dir))
            
            # Triângulo 2
            vertices.append((x_inf_esq, y_inf_esq, z_inf_esq))
            vertices.append((x_sup_dir, y_sup_dir, z_sup_dir))
            vertices.append((x_inf_dir, y_inf_dir, z_inf_dir))
            
    return vertices

In [283]:
def criar_lago():
    """Gera um disco azul no plano xz representando o lago."""
    return gerar_circulo(2.0, 32, centro_y=0.0)

def criar_barco():
    """Gera o casco do barco no formato de um prisma trapezoidal afinando nas pontas."""
    vertices = []
    comprimento = 0.5
    largura = 0.1
    altura = 0.15
    ponta = 0.08
    y_base = 0.0
    y_topo = altura

    base_esq_tras = (-largura, y_base, -comprimento * 0.8)
    base_dir_tras = ( largura, y_base, -comprimento * 0.8)
    base_dir_frente = ( largura * 0.3, y_base,  comprimento * 0.6)
    base_esq_frente = (-largura * 0.3, y_base,  comprimento * 0.6)

    topo_esq_tras = (-largura * 1.3, y_topo, -comprimento * 1.15)
    topo_dir_tras = ( largura * 1.3, y_topo, -comprimento * 1.15)
    topo_dir_frente = ( ponta,       y_topo,  comprimento * 1.1)
    topo_esq_frente = (-ponta,       y_topo,  comprimento * 1.1)

    vertices += [base_esq_tras, base_dir_tras, base_dir_frente,  base_esq_tras, base_dir_frente, base_esq_frente]
    vertices += [base_esq_tras, base_esq_frente, topo_esq_frente, base_esq_tras, topo_esq_frente, topo_esq_tras]
    vertices += [base_dir_tras, base_dir_frente, topo_dir_frente, base_dir_tras, topo_dir_frente, topo_dir_tras]
    vertices += [base_esq_tras, base_dir_tras, topo_dir_tras,   base_esq_tras, topo_dir_tras, topo_esq_tras]
    vertices += [base_esq_frente, base_dir_frente, topo_dir_frente, base_esq_frente, topo_dir_frente, topo_esq_frente]

    return vertices

def criar_mastro():
    """Gera um cilindro fino vertical representando o mastro do barco."""
    mastro = gerar_cilindro(raio=0.015, altura=0.7, n_segmentos=8, y_base=0.1)
    deslocamento_z = -0.13
    return [(x, y, z + deslocamento_z) for x, y, z in mastro]

def criar_vela():
    """Gera um triângulo no plano YZ simulando a vela."""
    deslocamento_z = -0.13
    topo = (0.0, 0.75, -0.02 + deslocamento_z)
    base_mastro = (0.0, 0.3, -0.02 + deslocamento_z)
    ponta_vela = (0.0, 0.3, -0.35 + deslocamento_z)     
    
    return [topo, base_mastro, ponta_vela]

def criar_pescador():
    """Gera a geometria do pescador composta por uma esfera (cabeça) e cilindros (corpo e membros)."""
    vertices_cabeca = gerar_esfera(raio=0.04, n_latitudes=8, n_longitudes=8, centro_y=0.32)
    vertices_corpo = gerar_cilindro(raio=0.035, altura=0.15, n_segmentos=8, y_base=0.15)
    
    braco_esquerdo = gerar_cilindro(raio=0.015, altura=0.12, n_segmentos=6, y_base=0.15)
    braco_direito = gerar_cilindro(raio=0.015, altura=0.12, n_segmentos=6, y_base=0.15)
    
    perna_esquerda = gerar_cilindro(raio=0.02, altura=0.12, n_segmentos=6, y_base=0.03)
    perna_direita = gerar_cilindro(raio=0.02, altura=0.12, n_segmentos=6, y_base=0.03)

    vertices_corpo += [(x - 0.05, y, z) for x, y, z in braco_esquerdo]
    vertices_corpo += [(x + 0.05, y, z) for x, y, z in braco_direito]
    vertices_corpo += [(x - 0.025, y, z) for x, y, z in perna_esquerda]
    vertices_corpo += [(x + 0.025, y, z) for x, y, z in perna_direita]

    return vertices_cabeca, vertices_corpo

def criar_bone():
    """Gera a geometria do boné do pescador (copa cilíndrica e aba plana)."""
    vertices = gerar_cilindro(raio=0.045, altura=0.02, n_segmentos=8, y_base=0.35)

    aba_y = 0.35
    aba_esq_base = (-0.04, aba_y, 0.03)
    aba_dir_base = ( 0.04, aba_y, 0.03)
    aba_esq_ponta = (-0.04, aba_y, 0.06)
    aba_dir_ponta = ( 0.04, aba_y, 0.06)
    
    vertices += [aba_esq_base, aba_dir_base, aba_esq_ponta]
    vertices += [aba_dir_base, aba_dir_ponta, aba_esq_ponta]
    vertices += [aba_esq_base, aba_esq_ponta, aba_dir_base]
    vertices += [aba_dir_base, aba_esq_ponta, aba_dir_ponta]
    
    return vertices

def criar_vara():
    """Gera um cilindro fino representando a vara de pesca."""
    return gerar_cilindro(raio=0.005, altura=0.4, n_segmentos=6, y_base=0.25)

def criar_linha_vara():
    """Gera as coordenadas para a linha de pesca (desenhada como GL_LINES)."""
    return [(0.0, 0.0, 0.0), (0.0, -0.65, 0.0)]

def criar_balde():
    """Gera os triângulos de um tronco de cone representando o balde de iscas."""
    vertices = []
    n_segmentos = 8
    raio_base = 0.03
    raio_topo = 0.04
    altura = 0.06
    y_base = 0.1
    y_topo = y_base + altura
    
    for indice_segmento in range(n_segmentos):
        angulo_inicial = 2.0 * math.pi * indice_segmento / n_segmentos
        angulo_final = 2.0 * math.pi * (indice_segmento + 1) / n_segmentos
        
        x_base_inicial = raio_base * math.cos(angulo_inicial)
        z_base_inicial = raio_base * math.sin(angulo_inicial)
        
        x_base_final = raio_base * math.cos(angulo_final)
        z_base_final = raio_base * math.sin(angulo_final)
        
        x_topo_inicial = raio_topo * math.cos(angulo_inicial)
        z_topo_inicial = raio_topo * math.sin(angulo_inicial)
        
        x_topo_final = raio_topo * math.cos(angulo_final)
        z_topo_final = raio_topo * math.sin(angulo_final)
        
        vertices.append((x_base_inicial, y_base, z_base_inicial))
        vertices.append((x_base_final, y_base, z_base_final))
        vertices.append((x_topo_inicial, y_topo, z_topo_inicial))
        
        vertices.append((x_base_final, y_base, z_base_final))
        vertices.append((x_topo_final, y_topo, z_topo_final))
        vertices.append((x_topo_inicial, y_topo, z_topo_inicial))
        
    vertices += gerar_circulo(raio_base, n_segmentos, y_base)
    return vertices

def criar_peixe():
    """Gera a geometria do peixe (elipse para o corpo, e triângulos para cauda e barbatana)."""
    vertices = []
    n_segmentos = 16
    raio_x = 0.08
    raio_z = 0.03
    altura_y = 0.001
    
    for indice_segmento in range(n_segmentos):
        angulo_inicial = 2.0 * math.pi * indice_segmento / n_segmentos
        angulo_final = 2.0 * math.pi * (indice_segmento + 1) / n_segmentos
        
        x_inicial = raio_x * math.cos(angulo_inicial)
        z_inicial = raio_z * math.sin(angulo_inicial)
        
        x_final = raio_x * math.cos(angulo_final)
        z_final = raio_z * math.sin(angulo_final)
        
        vertices.append((0.0, altura_y, 0.0))
        vertices.append((x_inicial, altura_y, z_inicial))
        vertices.append((x_final, altura_y, z_final))
        
    base_cauda = (-raio_x, altura_y, 0.0)
    ponta_cauda_dir = (-raio_x - 0.04, altura_y, 0.025)
    ponta_cauda_esq = (-raio_x - 0.04, altura_y, -0.025)
    
    vertices.append(base_cauda)
    vertices.append(ponta_cauda_dir)
    vertices.append(ponta_cauda_esq)
    
    base_barbatana = (0.0, altura_y, 0.0)
    topo_barbatana = (-0.02, altura_y + 0.03, 0.0)
    frente_barbatana = (0.02, altura_y, 0.0)
    
    vertices.append(base_barbatana)
    vertices.append(topo_barbatana)
    vertices.append(frente_barbatana)
    
    return vertices

In [284]:
quantidade_peixes = 50
vertices_peixes = [criar_peixe() for _ in range(quantidade_peixes)]
vertices_cabeca, vertices_corpo = criar_pescador()

# Definimos a cena em uma única estrutura unificada (Nome, Vértices, Cor, Modo de Desenho)
configuracao_cena = [
    ('lago',            criar_lago(),       (0.0, 0.4, 0.8, 1.0), GL_TRIANGLES),
    ('barco',           criar_barco(),      (0.55, 0.27, 0.07, 1.0), GL_TRIANGLES),
    ('mastro',          criar_mastro(),     (0.55, 0.27, 0.07, 1.0), GL_TRIANGLES),
    ('vela',            criar_vela(),       (0.95, 0.95, 0.95, 1.0), GL_TRIANGLES),
    ('pescador_cabeca', vertices_cabeca,    (0.87, 0.72, 0.53, 1.0), GL_TRIANGLES),
    ('pescador_corpo',  vertices_corpo,     (0.8, 0.1, 0.1, 1.0), GL_TRIANGLES),
    ('bone',            criar_bone(),       (0.1, 0.1, 0.8, 1.0), GL_TRIANGLES),
    ('vara',            criar_vara(),       (0.6, 0.5, 0.2, 1.0), GL_TRIANGLES),
    ('linha',           criar_linha_vara(), (0.1, 0.1, 0.1, 1.0), GL_LINES),
    ('balde',           criar_balde(),      (0.5, 0.5, 0.5, 1.0), GL_TRIANGLES),
    ('pescado',         criar_peixe(),      (0.25, 0.25, 0.25, 1.0), GL_TRIANGLES)
]

for i, peixe in enumerate(vertices_peixes):
    configuracao_cena.append((f'peixe{i+1}', peixe, (0.25, 0.25, 0.25, 1.0), GL_TRIANGLES))


def compilar_dados_da_cena(configuracoes):
    """
    Processa a lista de configurações, retornando o dicionário de metadados 
    (offsets e cores) e a lista linear de todos os vértices juntos.
    """
    metadados_objetos = {}
    todos_vertices = []
    offset_atual = 0

    for nome, verts, cor, modo in configuracoes:
        quantidade_vertices = len(verts)
        
        metadados_objetos[nome] = {
            'offset': offset_atual,
            'count': quantidade_vertices,
            'cor': cor,
            'modo': modo
        }
        
        todos_vertices.extend(verts)
        offset_atual += quantidade_vertices

    return metadados_objetos, todos_vertices


objetos, lista_total_vertices = compilar_dados_da_cena(configuracao_cena)

# Criação do array do NumPy
vertices = np.zeros(len(lista_total_vertices), [("position", np.float32, 3)])
vertices['position'] = lista_total_vertices

# Log de verificação
print(f"Total de vertices: {len(lista_total_vertices)}")
for nome, obj in objetos.items():
    print(f"  {nome}: offset={obj['offset']}, count={obj['count']}")

Total de vertices: 4001
  lago: offset=0, count=96
  barco: offset=96, count=30
  mastro: offset=126, count=96
  vela: offset=222, count=3
  pescador_cabeca: offset=225, count=384
  pescador_corpo: offset=609, count=384
  bone: offset=993, count=108
  vara: offset=1101, count=72
  linha: offset=1173, count=2
  balde: offset=1175, count=72
  pescado: offset=1247, count=54
  peixe1: offset=1301, count=54
  peixe2: offset=1355, count=54
  peixe3: offset=1409, count=54
  peixe4: offset=1463, count=54
  peixe5: offset=1517, count=54
  peixe6: offset=1571, count=54
  peixe7: offset=1625, count=54
  peixe8: offset=1679, count=54
  peixe9: offset=1733, count=54
  peixe10: offset=1787, count=54
  peixe11: offset=1841, count=54
  peixe12: offset=1895, count=54
  peixe13: offset=1949, count=54
  peixe14: offset=2003, count=54
  peixe15: offset=2057, count=54
  peixe16: offset=2111, count=54
  peixe17: offset=2165, count=54
  peixe18: offset=2219, count=54
  peixe19: offset=2273, count=54
  peixe2

In [285]:
def enviar_vertices_para_gpu(programa_shader, array_vertices):
    """
    Cria o Vertex Buffer Object (VBO), envia os dados para a placa de vídeo 
    e mapeia o atributo 'position' do shader.
    """
    id_buffer = glGenBuffers(1)
    glBindBuffer(GL_ARRAY_BUFFER, id_buffer)
    glBufferData(GL_ARRAY_BUFFER, array_vertices.nbytes, array_vertices, GL_STATIC_DRAW)

    passo_bytes = array_vertices.strides[0]
    loc_position = glGetAttribLocation(programa_shader, "position")
    
    glEnableVertexAttribArray(loc_position)
    glVertexAttribPointer(loc_position, 3, GL_FLOAT, False, passo_bytes, None)

    return id_buffer


def obter_variaveis_uniformes(programa_shader):
    """Retorna as localizações das variáveis de cor e transformação do shader."""
    loc_cor = glGetUniformLocation(programa_shader, "color")
    loc_matriz = glGetUniformLocation(programa_shader, "mat_transformation")
    
    return loc_cor, loc_matriz

enviar_vertices_para_gpu(program, vertices)
loc_color, loc_mat = obter_variaveis_uniformes(program)

In [286]:
class EstadoCena:
    """Armazena e gerencia todo o estado mutável da cena 3D."""
    
    def __init__(self, quantidade_peixes):
        # Estado do Barco
        self.barco_x = 0.0
        self.barco_z = 0.0
        self.barco_angulo = 0.0
        self.velocidade_barco = 0.05
        self.velocidade_rotacao = 0.05
        self.barco_onda_x = 0.0
        self.barco_onda_z = 0.0

        # Estado da Pescaria
        self.pescado_y = 0.0
        self.limite_altura_pesca = 0.5
        self.passo_pesca = 0.05

        # Estado dos Peixes
        self.peixe_escala = 1.0
        self.velocidade_exp_peixe = 1.0
        self.peixes = self._inicializar_peixes(quantidade_peixes)

        # Configurações de Renderização
        self.wireframe = False

    def _inicializar_peixes(self, quantidade):
        """Distribui os peixes em um círculo inicial."""
        peixes = []
        for i in range(quantidade):
            angulo = 2.0 * math.pi * i / quantidade
            peixes.append({
                'x': 0.5 * math.cos(angulo),
                'z': 0.5 * math.sin(angulo),
                'angulo': angulo
            })
        return peixes

    # --- Ações de Movimento ---
    
    def mover_barco(self, direcao):
        """Move o barco para frente (1) ou para trás (-1)."""
        self.barco_x += direcao * self.velocidade_barco * math.sin(self.barco_angulo)
        self.barco_z += direcao * self.velocidade_barco * math.cos(self.barco_angulo)

    def virar_barco(self, direcao):
        """Vira o barco para esquerda (1) ou direita (-1) com leve avanço."""
        self.barco_angulo += direcao * self.velocidade_rotacao
        self.barco_x += self.velocidade_barco * math.sin(self.barco_angulo) * 0.5
        self.barco_z += self.velocidade_barco * math.cos(self.barco_angulo) * 0.5

    def alterar_escala_peixes(self, incremento):
        """Aumenta ou diminui os peixes, respeitando limites máximo e mínimo."""
        self.peixe_escala += incremento
        self.peixe_escala = max(0.2, min(self.peixe_escala, 3.0))

    def recolher_linha(self, incremento):
        """Puxa ou solta o peixe fisgado, limitando a altura máxima e mínima."""
        self.pescado_y += incremento
        self.pescado_y = max(0.0, min(self.pescado_y, self.limite_altura_pesca))

    def alternar_wireframe(self):
        self.wireframe = not self.wireframe


# ==========================================
# Instanciamos o estado geral da cena
# ==========================================
estado = EstadoCena(quantidade_peixes=50)


def processar_entrada_teclado(window, key, scancode, action, mods):
    """Mapeia os inputs do usuário para ações no estado da cena."""
    
    # Ignora eventos que não sejam pressionar ou segurar a tecla
    if action != glfw.PRESS and action != glfw.REPEAT:
        return

    # Controles do Barco
    if key == glfw.KEY_W:
        estado.mover_barco(direcao=1)
    elif key == glfw.KEY_S:
        estado.mover_barco(direcao=-1)
    elif key == glfw.KEY_A:
        estado.virar_barco(direcao=1)
    elif key == glfw.KEY_D:
        estado.virar_barco(direcao=-1)
        
    # Controles dos Peixes
    elif key == glfw.KEY_Z:
        estado.alterar_escala_peixes(incremento=-0.1)
    elif key == glfw.KEY_X:
        estado.alterar_escala_peixes(incremento=0.1)
        
    # Controles da Vara de Pescar
    elif key == glfw.KEY_C:
        estado.recolher_linha(incremento=0.05)
    elif key == glfw.KEY_V:
        estado.recolher_linha(incremento=-0.05)
        
    # Controles Visuais
    elif key == glfw.KEY_P and action == glfw.PRESS:
        estado.alternar_wireframe()

glfw.set_key_callback(window, processar_entrada_teclado)

In [287]:
def mat_to_numpy(m):
    """Converte glm.mat4 para numpy array 16 floats."""
    return np.array(m, dtype=np.float32).flatten()

In [288]:
glfw.show_window(window)

In [289]:
# ==========================================
# 1. Lógica (Física e Atualizações)
# ==========================================
def atualizar_fisica_peixes(estado):
    """Atualiza a posição dos peixes e cuida para que não saiam do lago."""
    fator_movimento = np.power(0.001, estado.velocidade_exp_peixe)
    
    # Reseta a velocidade e espalha os peixes se estiverem muito lentos
    if fator_movimento < 0.000005:
        estado.velocidade_exp_peixe = 0.6
        for peixe in estado.peixes:
            peixe['angulo'] = np.random.uniform(0, 2 * np.pi)
    else:
        estado.velocidade_exp_peixe += 0.005
        
    limite_lago = 1.9 
    
    for peixe in estado.peixes:
        prox_x = peixe['x'] + fator_movimento * math.sin(peixe['angulo'])
        prox_z = peixe['z'] + fator_movimento * math.cos(peixe['angulo'])
        
        distancia = math.sqrt(prox_x**2 + prox_z**2)
        
        if distancia > limite_lago:
            peixe['angulo'] += math.pi + np.random.uniform(-0.5, 0.5)
        else:
            peixe['x'] = prox_x
            peixe['z'] = prox_z

# ==========================================
# 2. Hierarquia da Cena (Scene Graph)
# ==========================================
def calcular_matrizes_da_cena(estado, tempo):
    """Calcula as matrizes Model de todos os objetos baseados em suas dependências."""
    matrizes = {}
    
    # Lago (Fixo na origem)
    matrizes['lago'] = glm.mat4(1.0)

    # Barco (Base para o resto da pescaria)
    mat_barco = glm.mat4(1.0)
    mat_barco = glm.translate(mat_barco, glm.vec3(estado.barco_x, 0.05, estado.barco_z))
    mat_barco = glm.rotate(mat_barco, estado.barco_angulo, glm.vec3(0, 1, 0))
    
    # Oscilação do barco baseada no tempo
    onda_x = math.sin(tempo * 2.0) * 0.05
    onda_z = math.cos(tempo * 1.5) * 0.03
    mat_barco = glm.rotate(mat_barco, onda_x, glm.vec3(1, 0, 0))
    mat_barco = glm.rotate(mat_barco, onda_z, glm.vec3(0, 0, 1))

    matrizes['barco'] = mat_barco
    matrizes['mastro'] = mat_barco
    matrizes['vela'] = mat_barco
    matrizes['balde'] = mat_barco * glm.translate(glm.mat4(1.0), glm.vec3(-0.04, -0.06, -0.35))

    # Pescador (Filho do Barco)
    mat_pescador = mat_barco * glm.translate(glm.mat4(1.0), glm.vec3(0.0, 0.0, 0.05))
    mat_pescador = glm.rotate(mat_pescador, 1, glm.vec3(0, 1, 0))
    
    matrizes['pescador_cabeca'] = mat_pescador
    matrizes['pescador_corpo'] = mat_pescador
    matrizes['bone'] = mat_pescador

    # Vara (Filha do Pescador)
    mat_vara = mat_pescador * glm.translate(glm.mat4(1.0), glm.vec3(0.05, 0.0, -0.15))
    mat_vara = mat_vara * glm.rotate(glm.mat4(1.0), 0.75, glm.vec3(1, 0, 0))
    matrizes['vara'] = mat_vara

    # Linha (Filha da Vara)
    ponta_vara = mat_vara * glm.vec4(0.0, 0.65, 0.0, 1.0)
    mat_linha = glm.translate(glm.mat4(1.0), glm.vec3(ponta_vara.x, ponta_vara.y, ponta_vara.z))
    
    # Pescado (Filho da Linha)
    mat_pescado = mat_linha * glm.translate(glm.mat4(1.0), glm.vec3(0.0, estado.pescado_y - 0.65, 0.0))
    mat_pescado = glm.rotate(mat_pescado, tempo, glm.vec3(0, 1, 0))
    mat_pescado = glm.rotate(mat_pescado, np.pi/2, glm.vec3(0, 0, 1))
    matrizes['pescado'] = mat_pescado
    
    # Escala a linha depois de posicionar o pescado (para não deformar o peixe)
    escala_y_linha = 1.0 - abs(estado.pescado_y + 0.07) / 0.65
    matrizes['linha'] = glm.scale(mat_linha, glm.vec3(1.0, escala_y_linha, 1.0))

    # Peixes Livres (Independentes)
    for i, peixe in enumerate(estado.peixes):
        mat_peixe = glm.mat4(1.0)
        mat_peixe = glm.translate(mat_peixe, glm.vec3(peixe['x'], 0.0, peixe['z']))
        mat_peixe = glm.rotate(mat_peixe, peixe['angulo'] - np.pi/2, glm.vec3(0, 1, 0))
        mat_peixe = glm.scale(mat_peixe, glm.vec3(estado.peixe_escala))
        matrizes[f'peixe{i+1}'] = mat_peixe

    return matrizes

# ==========================================
# 3. Renderização (OpenGL)
# ==========================================
def preparar_frame(wireframe):
    """Limpa a tela e define o modo de polígono para o frame atual."""
    glClear(GL_COLOR_BUFFER_BIT | GL_DEPTH_BUFFER_BIT)
    glClearColor(0.2, 0.4, 0.15, 1.0)
    
    if wireframe:
        glPolygonMode(GL_FRONT_AND_BACK, GL_LINE)
    else:
        glPolygonMode(GL_FRONT_AND_BACK, GL_FILL)

def desenhar_objetos(matrizes, objetos_config, view_projection):
    """Envia as matrizes para a GPU e dispara as chamadas de desenho."""
    for nome, obj in objetos_config.items():
        mat_model = matrizes[nome]
        mat_final = view_projection * mat_model
        
        glUniformMatrix4fv(loc_mat, 1, GL_TRUE, mat_to_numpy(mat_final))
        glUniform4f(loc_color, *obj['cor'])
        glDrawArrays(obj['modo'], obj['offset'], obj['count'])


# ==========================================
# O Loop Principal
# ==========================================
glEnable(GL_DEPTH_TEST)

while not glfw.window_should_close(window):
    glfw.poll_events()
    tempo_atual = glfw.get_time()

    # 1. Update (A física acontece aqui)
    atualizar_fisica_peixes(estado)
    
    # 2. Scene Graph (As posições são calculadas aqui)
    matrizes_da_cena = calcular_matrizes_da_cena(estado, tempo_atual)

    # 3. Draw (Os pixels são pintados aqui)
    preparar_frame(estado.wireframe)
    desenhar_objetos(matrizes_da_cena, objetos, mat_vp)

    glfw.swap_buffers(window)

glfw.terminate()